# Naive Bayes
Accompanies the notes on **Naive Bayes**.

## 1. Multinomial NB: the five messages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB

docs = ["unauthorized charge fee", "stolen card charge fee", "fraud fee dispute", "late fee payment", "interest payment fee"]
labels = ["Fraud", "Fraud", "Fraud", "Billing", "Billing"]
vect = CountVectorizer()
counts = vect.fit_transform(docs)
query = vect.transform(["charge payment fee"])
mnb = MultinomialNB(alpha=1.0).fit(counts, labels)
print("classes:", mnb.classes_, " P(class | query):", mnb.predict_proba(query).round(4))

## 2. Gaussian NB by hand, checked

In [ ]:
from scipy.stats import norm

score0 = 0.75 * norm.pdf(30, 40, np.sqrt(100)) * norm.pdf(650, 720, np.sqrt(900))
score1 = 0.25 * norm.pdf(30, 26, np.sqrt(25)) * norm.pdf(650, 620, np.sqrt(1600))
print(f"Score(0) = {score0:.4e}, Score(1) = {score1:.4e}, P(default | x) = {score1 / (score0 + score1):.4f}")

## 3. Bernoulli NB on the same messages

In [ ]:
bnb = BernoulliNB(alpha=1.0).fit(counts > 0, labels)
print("P(class | query):", bnb.predict_proba(query > 0).round(4))

## 4. Gaussian NB vs logistic regression on the bank data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../../data/bd_train.csv")
y = (bd["Revenue.Grid"] == 1).astype(int)
numeric = bd.drop(columns=["REF_NO", "Revenue.Grid"]).select_dtypes("number").columns
X = np.log1p(bd[numeric].clip(lower=0))           # money columns are very skewed; logs are closer to normal
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
gnb = GaussianNB().fit(x_train, y_train)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(x_train, y_train)
for name, model in [("Gaussian NB", gnb), ("logistic regression", logit)]:
    p = model.predict_proba(x_test)[:, 1]
    print(f"{name:20s}: test AUC {roc_auc_score(y_test, p):.4f}, predictions below 0.01 or above 0.99: {np.mean((p < 0.01) | (p > 0.99)):.1%}")

## 5. How trustworthy are the probabilities?

In [ ]:
p_nb = gnb.predict_proba(x_test)[:, 1]
bins = pd.cut(p_nb, [0, 0.01, 0.5, 0.99, 1.0], include_lowest=True)
print(pd.DataFrame({"predicted": p_nb, "actual": y_test.values}).groupby(bins, observed=True)
      .agg(customers=("actual", "size"), mean_predicted=("predicted", "mean"), actual_rate=("actual", "mean")).round(3))

## 6. Classifying real complaints by product

In [ ]:
import time
cc = pd.read_csv("../../data/Consumer_Complaints_train.csv.zip", usecols=["Product", "Consumer complaint narrative"])
cc = cc.dropna().sample(20000, random_state=0)
text_train, text_test, prod_train, prod_test = train_test_split(cc["Consumer complaint narrative"], cc["Product"],
                                                                test_size=0.25, random_state=0, stratify=cc["Product"])
vect = CountVectorizer(stop_words="english", min_df=3)
w_train = vect.fit_transform(text_train); w_test = vect.transform(text_test)
print("documents:", w_train.shape[0], " vocabulary:", w_train.shape[1], " products:", cc["Product"].nunique())
for name, model in [("Multinomial NB", MultinomialNB(alpha=1.0)), ("Bernoulli NB", BernoulliNB(alpha=1.0)),
                    ("logistic regression", LogisticRegression(max_iter=3000, C=0.5))]:
    start = time.time(); model.fit(w_train, prod_train)
    print(f"{name:20s}: test accuracy {(model.predict(w_test) == prod_test).mean():.3f}, fit time {time.time() - start:.2f}s")

## 7. The most characteristic words per product

In [ ]:
nb = MultinomialNB(alpha=1.0).fit(w_train, prod_train)
words = np.array(vect.get_feature_names_out())
for cls in ["Mortgage", "Student loan", "Credit card"]:
    k = list(nb.classes_).index(cls)
    others = np.delete(nb.feature_log_prob_, k, axis=0).mean(axis=0)
    print(f"{cls:13s}:", ", ".join(words[np.argsort(nb.feature_log_prob_[k] - others)[-8:][::-1]]))

## Try it yourself
1. `MultinomialNB(alpha=a)` for a in 0.01, 0.1, 1, 3 in section 6.